In [1]:
import pandas as pd
import sqlite3
import json
import os

data_path = '/home/nororism/project/foodstuff/data/dump/meal_dump.json'
database_path = '/home/nororism/project/foodstuff/data/recipies.db'

con = sqlite3.connect(database_path)
cur = con.cursor()

In [ ]:
#enable foreign keys 
con.execute('PRAGMA foreign_keys = ON;')

In [37]:
cur.execute('PRAGMA foreign_keys = ON')

cur.executescript('''
    CREATE TABLE IF NOT EXISTS user (
        user_id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT NOT NULL UNIQUE,
        password TEXT NOT NULL,
        email TEXT NOT NULL UNIQUE
    );

    CREATE TABLE IF NOT EXISTS recipe (
        recipe_id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT,
        author VARCHAR(255),
        url VARCHAR(255),
        category VARCHAR(255),
        cuisine VARCHAR(255),
        description TEXT,
        image_url VARCHAR(255),
        total_time INTEGER,
        yields VARCHAR(255),
        ratings FLOAT,
        ratings_count INTEGER
    );

    CREATE TABLE IF NOT EXISTS ingredients (
        ingredient_id INTEGER PRIMARY KEY AUTOINCREMENT,
        ingredient TEXT UNIQUE
    );

    CREATE TABLE IF NOT EXISTS keywords (
        keyword_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name VARCHAR(255) UNIQUE
    );

    CREATE TABLE IF NOT EXISTS mealplan (
        plan_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        created TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (user_id) REFERENCES user(user_id)
    );

    CREATE TABLE IF NOT EXISTS user_likes (
        user_id INTEGER,
        recipe_id INTEGER,
        liked_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        PRIMARY KEY (user_id, recipe_id),
        FOREIGN KEY (user_id) REFERENCES user(user_id),
        FOREIGN KEY (recipe_id) REFERENCES recipe(recipe_id)
    );

    CREATE TABLE IF NOT EXISTS mealplan_recipe (
        plan_id INTEGER,
        recipe_id INTEGER,
        PRIMARY KEY (plan_id, recipe_id),
        FOREIGN KEY (plan_id) REFERENCES mealplan(plan_id),
        FOREIGN KEY (recipe_id) REFERENCES recipe(recipe_id)
    );

    CREATE TABLE IF NOT EXISTS recipe_ingredients (
        recipe_id INTEGER,
        ingredient_id INTEGER,
        PRIMARY KEY (recipe_id, ingredient_id),
        FOREIGN KEY (recipe_id) REFERENCES recipe(recipe_id),
        FOREIGN KEY (ingredient_id) REFERENCES ingredients(ingredient_id)
    );

    CREATE TABLE IF NOT EXISTS instructions (
        instruction_id INTEGER PRIMARY KEY AUTOINCREMENT,
        recipe_id INTEGER,
        step INTEGER,
        text TEXT,
        FOREIGN KEY (recipe_id) REFERENCES recipe(recipe_id)
    );

    CREATE TABLE IF NOT EXISTS nutrients (
        recipe_id INTEGER PRIMARY KEY,
        calories INTEGER,
        protein INTEGER,
        carbs INTEGER,
        fats INTEGER,
        fiber INTEGER,
        sodium INTEGER,
        cholesterol INTEGER,
        FOREIGN KEY (recipe_id) REFERENCES recipe(recipe_id)
    );

    CREATE TABLE IF NOT EXISTS recipe_keywords (
        recipe_id INTEGER,
        keyword_id INTEGER,
        PRIMARY KEY (recipe_id, keyword_id),
        FOREIGN KEY (recipe_id) REFERENCES recipe(recipe_id),
        FOREIGN KEY (keyword_id) REFERENCES keywords(keyword_id)
    );

    CREATE TABLE IF NOT EXISTS grocery_list (
        list_id INTEGER PRIMARY KEY AUTOINCREMENT,
        plan_id INTEGER,
        generated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (plan_id) REFERENCES mealplan(plan_id)
    );

    CREATE TABLE IF NOT EXISTS grocery_list_items (
        list_id INTEGER,
        ingredient_id INTEGER,
        total_quantity DECIMAL(8,2),
        unit VARCHAR(30),
        checked BOOLEAN DEFAULT 0,
        PRIMARY KEY (list_id, ingredient_id),
        FOREIGN KEY (list_id) REFERENCES grocery_list(list_id),
        FOREIGN KEY (ingredient_id) REFERENCES ingredients(ingredient_id)
    );
''')

con.commit()

In [ ]:
cur.execute(
'''
    alter table mealplan add column if not exists name varchar(255) default ;
'''
)


In [6]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)

,name
0,grocery_list
1,grocery_list_items
2,ingredients
3,instructions
4,keywords
5,mealplan
6,mealplan_recipe
7,nutrients
8,recipe
9,recipe_ingredients


In [29]:
import json

def parse_nutrient(value):
    if value is None:
        return None
    # strips " g" or any non-numeric characters, returns int
    return int(''.join(filter(str.isdigit, str(value)))) or None

def import_recipe(cur, r):
    # Insert recipe
    cur.execute('''
        INSERT INTO recipe (title, author, url, category, cuisine, description, image_url, total_time, yields, ratings, ratings_count)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        r.get('title'),
        r.get('author'),
        r.get('canonical_url'),
        r.get('category'),
        r.get('cuisine'),
        r.get('description'),
        r.get('image'),
        r.get('total_time'),
        r.get('yields'),
        r.get('ratings'),
        r.get('ratings_count')
    ))
    recipe_id = cur.lastrowid

    # Insert ingredients
    for ingredient in r.get('ingredients', []):
        cur.execute('INSERT OR IGNORE INTO ingredients (ingredient) VALUES (?)', (ingredient,))
        cur.execute('SELECT ingredient_id FROM ingredients WHERE ingredient = ?', (ingredient,))
        ingredient_id = cur.fetchone()[0]
        cur.execute('INSERT OR IGNORE INTO recipe_ingredients (recipe_id, ingredient_id) VALUES (?, ?)', (recipe_id, ingredient_id))

    # Insert instructions
    for step, text in enumerate(r.get('instructions_list', []), start=1):
        cur.execute('INSERT INTO instructions (recipe_id, step, text) VALUES (?, ?, ?)', (recipe_id, step, text))

    # Insert keywords
    for keyword in r.get('keywords', []):
        cur.execute('INSERT OR IGNORE INTO keywords (name) VALUES (?)', (keyword,))
        cur.execute('SELECT keyword_id FROM keywords WHERE name = ?', (keyword,))
        keyword_id = cur.fetchone()[0]
        cur.execute('INSERT OR IGNORE INTO recipe_keywords (recipe_id, keyword_id) VALUES (?, ?)', (recipe_id, keyword_id))

    # Insert nutrients
    nutrients = r.get('nutrients')
    if nutrients:
        cur.execute('''
            INSERT OR IGNORE INTO nutrients (recipe_id, calories, protein, carbs, fats, fiber, sodium, cholesterol)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            recipe_id,
            parse_nutrient(nutrients.get('calories')),
            parse_nutrient(nutrients.get('proteinContent')),
            parse_nutrient(nutrients.get('carbohydrateContent')),
            parse_nutrient(nutrients.get('fatContent')),
            parse_nutrient(nutrients.get('fiberContent')),
            parse_nutrient(nutrients.get('sodiumContent')),
            parse_nutrient(nutrients.get('cholesterolContent'))
        ))

    return recipe_id

In [38]:
# Check for duplicates in the JSON file
with open(data_path) as f:
    recipes = json.load(f)

urls = [r.get('canonical_url') for r in recipes]
print(f"Total recipes: {len(recipes)}")
print(f"Unique recipes: {len(set(urls))}")

# Show which ones are duplicated
from collections import Counter
dupes = [url for url, count in Counter(urls).items() if count > 1]
print(f"Duplicated urls: {dupes}")


Total recipes: 1944
Unique recipes: 972
Duplicated urls: ['https://www.americastestkitchen.com/recipes/16142-shrimp-with-black-bean-sauce-for-two', 'https://www.americastestkitchen.com/recipes/16790-pan-seared-halibut-with-wilted-bitter-salad', 'https://www.americastestkitchen.com/recipes/17481-raspberry-mousse', 'https://www.americastestkitchen.com/recipes/16796-cast-iron-blueberry-cardamom-buttermilk-cake', 'https://www.americastestkitchen.com/recipes/17361-bacon-and-gruyere-egg-bites', 'https://www.americastestkitchen.com/recipes/14149-cheddar-crusted-grilled-cheese', 'https://www.americastestkitchen.com/recipes/16851-supercrunchy-buttermilk-fried-chicken', 'https://www.americastestkitchen.com/recipes/11920-japanese-steakhouse-steak-and-vegetables', 'https://www.americastestkitchen.com/recipes/17323-pork-tenderloin-medallions-with-lemon-parsley-pan-sauce', 'https://www.americastestkitchen.com/recipes/8973-one-pan-steak-with-sweet-potatoes-and-scallions', 'https://www.americastestkit

In [39]:
seen = set()
unique_recipes = []
for r in recipes:
    url = r.get('canonical_url')
    if url not in seen:
        seen.add(url)
        unique_recipes.append(r)

print(f"Importing {len(unique_recipes)} unique recipes")

for r in unique_recipes:
    import_recipe(cur, r)

con.commit()

Importing 972 unique recipes


In [40]:
pd.read_sql('SELECT * FROM recipe', con)

,recipe_id,title,author,url,category,cuisine,description,image_url,total_time,yields,ratings,ratings_count
0,1,Shrimp with Black Bean Sauce For Two,Erica Turner,https://www.americastestkitchen.com/recipes/16...,Main Courses,Chinese,Savor the rich flavors of our Shrimp with Blac...,https://res.cloudinary.com/hksqkdlah/image/upl...,30.0,2 servings,4.21,17.0
1,2,Pan-Seared Halibut with Wilted Bitter Salad,Ben Mims,https://www.americastestkitchen.com/recipes/16...,Main Courses,,Based on a dish we've seen on restaurant menus...,https://res.cloudinary.com/hksqkdlah/image/upl...,60.0,4 servings,4.06,13.0
2,3,Raspberry Mousse,Steve Dunn,https://www.americastestkitchen.com/recipes/17...,Desserts or Baked Goods,,Skipping fresh fruit means this mousse can be ...,https://res.cloudinary.com/hksqkdlah/image/upl...,345.0,6 servings,4.10,45.0
3,4,Cast Iron Blueberry-Cardamom Buttermilk Cake,Nicole Konstantinakos,https://www.americastestkitchen.com/recipes/16...,Desserts or Baked Goods,,"A simple, rustic berry cake that shines on any...",https://res.cloudinary.com/hksqkdlah/image/upl...,105.0,8 servings,4.13,16.0
4,5,Bacon and Gruyère Egg Bites,David Yu,https://www.americastestkitchen.com/recipes/17...,,,"To make this popular portable breakfast, we sm...",https://res.cloudinary.com/hksqkdlah/image/upl...,60.0,12 servings,4.35,154.0
...,...,...,...,...,...,...,...,...,...,...,...,...
967,968,Chicken Marsala,America's Test Kitchen,https://www.americastestkitchen.com/recipes/32...,Main Courses,ItalianEurope,"Chicken Marsala seems like a simple dish, but ...",https://res.cloudinary.com/hksqkdlah/image/upl...,NaN,4 servings,4.17,41.0
968,969,Slow-Roasted Chicken,America's Test Kitchen,https://www.americastestkitchen.com/recipes/12...,Main Courses,,"After roasting 14 birds, we offer our two favo...",https://res.cloudinary.com/hksqkdlah/image/upl...,155.0,4 servings,4.17,50.0
969,970,"Angel Hair Pasta with Basil, Caper, and Lemon ...",Steve Dunn,https://www.americastestkitchen.com/recipes/12...,Main Courses,ItalianEurope,"For perfectly al dente strands, throw out the ...",https://res.cloudinary.com/hksqkdlah/image/upl...,30.0,4 servings,4.16,88.0
970,971,The Best Gluten-Free Pizza,Andrew Janjigian,https://www.americastestkitchen.com/recipes/78...,Main Courses,,This gluten-free pizza recipe offers a delicio...,https://res.cloudinary.com/hksqkdlah/image/upl...,270.0,4 servings,4.16,129.0


In [41]:
pd.read_sql('''
    SELECT r.title, i.ingredient
    FROM recipe r
    JOIN recipe_ingredients ri ON r.recipe_id = ri.recipe_id
    JOIN ingredients i ON i.ingredient_id = ri.ingredient_id
    WHERE r.title = "Shrimp with Black Bean Sauce For Two"
''', con)

,title,ingredient
0,Shrimp with Black Bean Sauce For Two,1½ teaspoons soy sauce
1,Shrimp with Black Bean Sauce For Two,1½ teaspoons Shaoxing wine or dry sherry
2,Shrimp with Black Bean Sauce For Two,8 ounces extra-large shrimp (21 to 25 per poun...
3,Shrimp with Black Bean Sauce For Two,⅓ cup chicken broth
4,Shrimp with Black Bean Sauce For Two,1½ teaspoons oyster sauce
5,Shrimp with Black Bean Sauce For Two,1½ teaspoons sugar
6,Shrimp with Black Bean Sauce For Two,1 teaspoon toasted sesame oil
7,Shrimp with Black Bean Sauce For Two,¾ teaspoon cornstarch
8,Shrimp with Black Bean Sauce For Two,1 tablespoon vegetable oil
9,Shrimp with Black Bean Sauce For Two,"2½ tablespoons fermented black beans (dau si),..."


In [20]:
cur.executescript('''
    DELETE FROM recipe_ingredients;
    DELETE FROM recipe_keywords;
    DELETE FROM instructions;
    DELETE FROM nutrients;
    DELETE FROM ingredients;
    DELETE FROM keywords;
    DELETE FROM recipe;
''')
con.commit()

In [36]:
cur.executescript('''
    DROP TABLE IF EXISTS grocery_list_items;
    DROP TABLE IF EXISTS grocery_list;
    DROP TABLE IF EXISTS recipe_keywords;
    DROP TABLE IF EXISTS recipe_ingredients;
    DROP TABLE IF EXISTS instructions;
    DROP TABLE IF EXISTS nutrients;
    DROP TABLE IF EXISTS mealplan_recipe;
    DROP TABLE IF EXISTS user_likes;
    DROP TABLE IF EXISTS mealplan;
    DROP TABLE IF EXISTS recipe;
    DROP TABLE IF EXISTS ingredients;
    DROP TABLE IF EXISTS keywords;
    DROP TABLE IF EXISTS user;
''')
con.commit()

In [42]:
def search_by_tag(cur, tag):
    results = cur.execute('''
        SELECT r.recipe_id, r.title, r.category, r.cuisine, r.total_time, r.ratings
        FROM recipe r
        JOIN recipe_keywords rk ON r.recipe_id = rk.recipe_id
        JOIN keywords k ON k.keyword_id = rk.keyword_id
        WHERE k.name LIKE ?
        ORDER BY r.ratings DESC
    ''', (f'%{tag}%',)).fetchall()
    
    return results

In [43]:
results = search_by_tag(cur, 'Seafood')
for r in results:
    print(r)

(396, 'Sheet Pan Salmon with Mushroom Arugula Salad', 'Main Courses', '', 40, 5.0)
(803, 'Seafood Risotto with Shrimp, Mussels, and Squid', 'Main Courses', '', 90, 4.88)
(30, 'Broiled Hot Honey Salmon with Cucumber Salad', 'Main Courses', '', 50, 4.68)
(283, 'Spicy Salmon Sushi Bowls', 'Main Courses', 'Japanese', 45, 4.67)
(399, 'Salmon Toasts with Calabrian Chile–Garlic Butter', 'Main Courses', '', 35, 4.67)
(152, 'Maple-Soy Salmon Bowls with Quinoa and Brussels Sprouts', 'Main Courses', '', 35, 4.64)
(807, 'Halibut Puttanesca', 'Main Courses', 'Italian', 40, 4.64)
(491, 'Quick Pantry Clam Chowder', 'Main Courses', 'New England', 45, 4.55)
(492, "Shrimp Po' Boys", 'Main Courses', 'Creole & Cajun', 90, 4.54)
(493, 'Seafood Fra Diavolo', 'Main Courses', '', 55, 4.52)
(635, 'Monterey Bay Cioppino', 'Main Courses', 'California', 90, 4.51)
(496, 'Fried Calamari', 'Appetizers', 'Italian', 50, 4.5)
(637, 'Spiced Red Snapper with Creamy Tahini Sauce and Butter-Toasted Almonds', 'Main Courses'

In [ ]:
# # Add a test user
# cur.execute('''
#     INSERT INTO user (username, password, email) 
#     VALUES (?, ?, ?)
# ''', ('testuser', 'password123', 'test@email.com'))
user_id = cur.lastrowid

# # Create a meal plan
# cur.execute('''
#     INSERT INTO mealplan (user_id) VALUES (?)
# ''', (user_id,))
plan_id = cur.lastrowid

# Add a recipe to the plan
cur.execute('''
    INSERT INTO mealplan_recipe (plan_id, recipe_id) VALUES (?, ?)
''', (plan_id, 2))

# Like a recipe
cur.execute('''
    INSERT INTO user_likes (user_id, recipe_id) VALUES (?, ?)
''', (user_id, 2))

con.commit()

IntegrityError: UNIQUE constraint failed: mealplan_recipe.plan_id, mealplan_recipe.recipe_id

In [45]:
pd.read_sql('SELECT recipe_id, title FROM recipe LIMIT 10', con)

,recipe_id,title
0,1,Shrimp with Black Bean Sauce For Two
1,2,Pan-Seared Halibut with Wilted Bitter Salad
2,3,Raspberry Mousse
3,4,Cast Iron Blueberry-Cardamom Buttermilk Cake
4,5,Bacon and Gruyère Egg Bites
5,6,Cheddar-Crusted Grilled Cheese
6,7,Supercrunchy Buttermilk Fried Chicken
7,8,Japanese Steakhouse Steak and Vegetables
8,9,Pork Tenderloin Medallions with Lemon-Parsley ...
9,10,One-Pan Steak with Sweet Potatoes and Scallions


In [52]:
# Check user
pd.read_sql('SELECT * FROM user', con)

,user_id,username,password,email
0,1,testuser,password123,test@email.com


In [53]:
# Check mealplan
pd.read_sql('SELECT * FROM mealplan', con)

,plan_id,user_id,created
0,1,1,2026-04-21 17:28:13


In [54]:
# Check recipes in the meal plan
pd.read_sql('''
    SELECT u.username, mp.plan_id, r.title
    FROM user u
    JOIN mealplan mp ON u.user_id = mp.user_id
    JOIN mealplan_recipe mpr ON mp.plan_id = mpr.plan_id
    JOIN recipe r ON r.recipe_id = mpr.recipe_id
''', con)

,username,plan_id,title
0,testuser,1,Pan-Seared Halibut with Wilted Bitter Salad


In [55]:
# Check liked recipes
pd.read_sql('''
    SELECT u.username, r.title, ul.liked_at
    FROM user u
    JOIN user_likes ul ON u.user_id = ul.user_id
    JOIN recipe r ON r.recipe_id = ul.recipe_id
''', con)

,username,title,liked_at
0,testuser,Pan-Seared Halibut with Wilted Bitter Salad,2026-04-21 17:30:33


In [62]:
def generate_grocery_list(cur, plan_id):
    # Create grocery list for the plan
    cur.execute('''
        INSERT INTO grocery_list (plan_id) VALUES (?)
    ''', (plan_id,))
    list_id = cur.lastrowid

    # Get all ingredients from all recipes in the plan
    # INSERT OR IGNORE handles duplicate ingredients across recipes
    cur.execute('''
        INSERT OR IGNORE INTO grocery_list_items (list_id, ingredient_id, checked)
        SELECT DISTINCT ?, i.ingredient_id, 0
        FROM mealplan_recipe mpr
        JOIN recipe_ingredients ri ON mpr.recipe_id = ri.recipe_id
        JOIN ingredients i ON i.ingredient_id = ri.ingredient_id
        WHERE mpr.plan_id = ?
    ''', (list_id, plan_id))

    con.commit()
    return list_id

def view_grocery_list(list_id):
    return pd.read_sql('''
        SELECT 
            r.title as recipe,
            i.ingredient,
            CASE WHEN gli.checked = 1 THEN 'yes' ELSE 'no' END as checked
        FROM grocery_list_items gli
        JOIN ingredients i ON i.ingredient_id = gli.ingredient_id
        JOIN recipe_ingredients ri ON i.ingredient_id = ri.ingredient_id
        JOIN recipe r ON r.recipe_id = ri.recipe_id
        JOIN mealplan_recipe mpr ON r.recipe_id = mpr.recipe_id
        JOIN grocery_list gl ON gl.list_id = gli.list_id
        WHERE gli.list_id = ?
        AND mpr.plan_id = gl.plan_id
        ORDER BY r.title, i.ingredient
    ''', con, params=(list_id,))

In [64]:
# cur.execute('''
#     INSERT INTO mealplan_recipe (plan_id, recipe_id) VALUES (?, ?)
# ''', (plan_id, 3))

# con.commit()

# Generate the list
list_id = generate_grocery_list(cur, plan_id)
print(f"Grocery list created with id: {list_id}")

# View the list
view_grocery_list(list_id)

Grocery list created with id: 4


,recipe,ingredient,checked
0,Pan-Seared Halibut with Wilted Bitter Salad,1 head Belgian endive (4 ounces),no
1,Pan-Seared Halibut with Wilted Bitter Salad,"1 large shallot, minced",no
2,Pan-Seared Halibut with Wilted Bitter Salad,1 small fennel bulb,no
3,Pan-Seared Halibut with Wilted Bitter Salad,1 small head radicchio (6 ounces),no
4,Pan-Seared Halibut with Wilted Bitter Salad,2 ounces (2 cups) baby arugula,no
5,Pan-Seared Halibut with Wilted Bitter Salad,2 tablespoons Dijon mustard,no
6,Pan-Seared Halibut with Wilted Bitter Salad,"3 garlic cloves, minced",no
7,Pan-Seared Halibut with Wilted Bitter Salad,3 tablespoons water,no
8,Pan-Seared Halibut with Wilted Bitter Salad,"4 (6- to 8-ounce) skinless halibut fillets, 1 ...",no
9,Pan-Seared Halibut with Wilted Bitter Salad,"7 tablespoons extra-virgin olive oil, divided,...",no


In [7]:
pd.read_sql('''select * from user''', con)


,user_id,username,password,email
0,1,testuser,password123,test@email.com
